# BrDR and BrM Clients

**Who this is for.** Rating engineers who want AASHTOWare BrDR (BrR) results as
DataFrames instead of report printouts — and anyone wiring bridge data toward
AASHTOWare BrM.

**What it shows.** Direct SQL access to a BrDR Oracle database, the shipped
ODOT table map that makes those queries writable, and the rating-ratio engine
that predicts legal-load RFs from the vehicles you already rated.

**Prev:** [13. AISC Steel Sections and NSBA Splice Design](13.%20AISC%20Steel%20Sections%20and%20NSBA%20Splice%20Design.ipynb)
&nbsp;&middot;&nbsp; **Next:** [15. MIDAS Civil NX API ➡️](15.%20MIDAS%20Civil%20NX%20API.ipynb)

## Step 1 — Connect to the BrDR database

BrDR keeps everything in Oracle. `connect_to_brr()` reads `~/secrets.json`
(keys `BRR_USN`, `BRR_PASS`, `BRR_SERVER`, `BRR_PORT`, `BRR_SERVICE`) and hands
back a live SQLAlchemy connection + engine. You must be on a network that can
see the database server.

In [1]:
import os, sys
import pandas as pd

sys.path.insert(0, os.path.abspath("."))   # run from the civilpy repo root

from src.civilpy.structural.aashtoware.brr import connect_to_brr

try:
    conn, engine = connect_to_brr()
    bridges = pd.read_sql("SELECT bridge_id, nbi_structure_id, name FROM abw_bridge", conn)
    print(f"connected — {len(bridges)} bridges in this BrDR instance")
except Exception as e:
    print("BrDR database not reachable from here — run this on the agency network.")
    print(f"({type(e).__name__})")

BrDR database not reachable from here — run this on the agency network.
(KeyError)


## Step 2 — Know the schema before you query it

`AASHTOWARE_TABLES` is a shipped reference of the `ABW_*` tables observed in
ODOT's deployment — columns, whether the table is populated, row counts. Use it
to find where a value lives before writing SQL:

In [2]:
from src.civilpy.structural.aashtoware.odot_tables import AASHTOWARE_TABLES

print(f"{len(AASHTOWARE_TABLES)} ABW_* tables mapped")

# where do rating results live?
hits = [t for t, v in AASHTOWARE_TABLES.items()
        if any("rating" in c for c in v["Columns"])]
hits[:10]

918 ABW_* tables mapped


['ABW_EVENT_VEHICLE_TEMPLATE',
 'ABW_PS_PRECAST_BEAM_DEF',
 'ABW_FLNG_LAT_BEND_STRESS',
 'ABW_EVENT_VEHICLE',
 'ABW_LIB_VEHICLE',
 'ABW_SYS_DATABASE',
 'ABW_RATING_RESULTS',
 'ABW_RATING_RESULTS_SUMMARY',
 'ABW_STL_BEAM_DEF',
 'ABW_SUPER_STRUCT_DEF']

In [3]:
AASHTOWARE_TABLES["ABW_BRIDGE"]["Columns"][:15]

['bridge_id',
 'bridge_guid',
 'agency_code',
 'struct_num',
 'name',
 'bridge_rating_ind',
 'bridge_design_ind',
 'bridge_management_ind',
 'descr',
 'elevation',
 'x_plane_coordinate',
 'y_plane_coordinate',
 'prev_count_year',
 'recent_count_year',
 'recent_count_adtt']

## Step 3 — Predict legal-load RFs from the ratings you have

`civilpy.structural.rating_ratios` implements the demand-ratio method: your
known rating factors (HS20, HL-93, whatever BrDR produced) scale through
influence-line demand ratios to the vehicles you *didn't* run. Build a demand
basis for the span, then predict:

In [4]:
from src.civilpy.structural.rating_ratios import (
    identify_governing_case, predict_rating_factors, simple_span_demands,
)

basis = simple_span_demands(span=120.0)        # moments/shears for 13 vehicles
known = {"HS20": 1.08, "HL-93": 0.96, "SU4": 1.21}

preds = predict_rating_factors(known, basis, "positive_moment")
pd.DataFrame([{
    "vehicle": p.vehicle, "predicted RF": round(p.rf, 2),
    "from": ", ".join(f"{k}:{v:.2f}" for k, v in p.per_known.items()),
} for p in preds.values()])

,vehicle,predicted RF,from
0,Type 3,1.67,"HS20:1.51, HL-93:2.16, SU4:1.34"
1,Type 3S2,1.33,"HS20:1.20, HL-93:1.72, SU4:1.06"
2,Type 3-3,1.29,"HS20:1.17, HL-93:1.67, SU4:1.03"


`three_span_demands` does the same for continuous units (positive + negative
regions), and `identify_governing_case` tells you which action/region the
prediction should be based on. First statewide validation of this method came
in at a median |error| of ~2%.

## Step 4 — The vehicles behind the demands

The rating vehicles themselves (axle weights and spacings for HL-93, HS20, the
Ohio legal 2F1/3F1/4F1/5C1 trucks, the SU series, Type 3 units…) live in
`civilpy.structural.aashto.vehicles` — the same definitions feed the MIDAS
moving-load builders in notebook 15.

In [5]:
from src.civilpy.structural.aashto.vehicles import RATING_VEHICLES

su4 = RATING_VEHICLES["SU4"]
print(su4)
print("GVW:", su4.gvw_tons, "tons | axle spacings (ft):", su4.axle_spacings_ft)
pd.DataFrame([{"vehicle": k, "GVW (tons)": v.gvw_tons,
               "wheelbase (ft)": v.wheelbase_ft}
              for k, v in RATING_VEHICLES.items()])

RatingVehicle(SU4: 54 kip GVW, 4 axles over 18 ft)
GVW: 27.0 tons | axle spacings (ft): (10.0, 4.0, 4.0)


,vehicle,GVW (tons),wheelbase (ft)
0,HS20,36.00,28.000
1,HL-93,36.00,28.000
2,Type 3,25.00,19.000
3,Type 3S2,36.00,41.000
4,Type 3-3,40.00,54.000
5,SU4,27.00,18.000
6,SU5,31.00,22.000
7,SU6,34.75,26.000
8,SU7,38.75,30.000
9,EV2,28.75,15.000


## Step 5 — BrM (AASHTOWare Bridge Management)

BrM is a different product (ProMiles/Mayvue — *not* Bentley) with a REST
`/openapi/` surface secured by client-credentials (`BRM_CLIENT_ID` /
`BRM_KEY`). civilpy doesn't ship a BrM client — the working sync client and the
vendored official OpenAPI specs live in the (private) snbi_ui repo — but the
BrDR schema map above already shows the seam between the two products: BrDR's
own tables carry the BrM sync-endpoint configuration.

In [6]:
brm_cols = sorted({c for v in AASHTOWARE_TABLES.values()
                   for c in v["Columns"] if c.startswith("brm")})
brm_cols

['brm_data_column',
 'brm_data_table',
 'brm_default_data_column',
 'brm_default_data_table',
 'brm_endpoint_db_id',
 'brm_endpoint_ignore_cert_ind',
 'brm_endpoint_url',
 'brm_installed_ind',
 'brm_sync_setting_id',
 'brm_sync_setting_value_id']

**Related:** notebook *LRFD Girder Rating Walkthrough* rates a girder with the
civilpy LRFD engine; *AASHTO LRFD Distribution Factors and LRFR* and *AASHTO
MBE* cover the rating-side spec articles.